In [1]:
%cd ../../../

/Users/hoangle/Projects/untangling-people/Food-Waste-Optimization


In [2]:
import re
import sys
from itertools import zip_longest

import numpy as np
import pandas as pd
from loguru import logger


In [3]:
logger.remove()
logger.add(sys.stdout, level="DEBUG")

1

# Add meals

In [4]:
def hamming_distance(s1: str, s2: str) -> float:
    min_len = min(len(s1), len(s2))
    dist = sum(c1 != c2 for c1, c2 in zip_longest(s1, s2))  * 1.0 / min_len

    return dist

In [5]:
meals = {}

map_mealname2id = {}

## Process meal list

In [6]:
meals_raw = pd.read_excel("data/processed/phase_4/menus.xlsx", sheet_name="meals")

meals_raw.head()

,meal_code,meal_name,category,CO2
0,34.0,Sitruunaiset kalapaloja ja kukkakaalitsatsikia,kala,0.81
1,710.0,Broileri-Caesarsalaatti,kana,0.67
2,724.0,Broilerilasagnette,kana,0.82
3,725.0,"Broilerinuggetit, currykastiketta",kana,1.06
4,726.0,"Broileripyörykät, currykastike",kana,0.86


In [7]:
# Remove duplicate entries
meal_list = (
    meals_raw
    .dropna(axis=0, how='any')
    .drop(columns='CO2')
    .groupby('meal_code')
    .last()
    .reset_index()
    .rename(columns={
        'meal_code': 'meal_id',
        'meal_name': 'meal',
        'category': 'meal_type',
    })
)


# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({'meal': meal_name, 'tag': tag})
    
meal_list['meal'] = meal_list['meal'].str.strip().apply(_f_extract_tag)['meal']
meal_list = meal_list.groupby('meal').first().reset_index()


meal_list['meal_type'] = meal_list['meal_type'].str.strip().map({
    'kala': 'fish',
    'liha': 'meat',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kasvis': 'vegetarian',
    'keskiarvo': 'buffet',
})

meal_list['meal_id'] = meal_list['meal_id'].astype(int)

meal_list = meal_list.sort_values('meal_id').groupby('meal').first().reset_index()

meal_list = meal_list[~meal_list['meal'].str.lower().str.contains('take away')]

meal_list['schoolyear'] = "23-24"


meal_list.head()

,meal,meal_id,meal_type,schoolyear
0,"""Butter"" härkäpapua & pähkinää",9017,vegan,23-24
1,2023 Härkäpu-sienilasagnette,7201,vegan,23-24
2,Appelisiini-luomukikhernecurrya,9032,vegan,23-24
3,Artisokkavugetteja & tuoretomaattisalsaa,9102,vegan,23-24
4,Aurajuusto-pinaattilasagnette,7010,vegetarian,23-24


### Add meals in `meal_list` to dict `meals`

In [8]:
THRES = 0.1

for meal_other in meal_list.itertuples():
    if meal_other.meal in meals:    # exact match
        # With meal_list, we dont care exact match case
        logger.debug("here 1")
    else:
        # Find approximate match
        meal_id_best = None
        dist_best = 1e10
        for meal_name, meal_id in map_mealname2id.items():
            dist = hamming_distance(meal_other.meal, meal_name)

            if dist <= THRES and dist < dist_best:
                dist_best = dist
                meal_id_best = meal_id

        # Handle 2 cases: found approximate match and not found
        if meal_id_best is None:   # not found
            meals[meal_other.meal_id] = {
                'meal_type': meal_other.meal_type,
                'schoolyear': meal_other.schoolyear,
                'restaurant': None,
                'attributes': []
            }

            map_mealname2id[meal_other.meal] = meal_other.meal_id
        else:                   # found approximate match
            logger.debug(f"herer 2: {meal_other.meal_id} - {meal_id_best}")

            map_mealname2id[meal_other.meal] = meal_id_best

2025-01-30 12:40:19.712 | DEBUG    | __main__:<module>:29 - herer 2: 3656 - 6356
2025-01-30 12:40:19.752 | DEBUG    | __main__:<module>:29 - herer 2: 1125 - 6963
2025-01-30 12:40:19.882 | DEBUG    | __main__:<module>:29 - herer 2: 6045 - 6044


# Merge meals in menu to `meals`

## Process menu file

In [9]:
path = "data/processed/phase_4/menus.xlsx"

weeks = ["week1", "week2", "week3", "week4", "week5", "week6"]

list_menus = []
for week in weeks:
    raw = pd.read_excel(path, sheet_name=week)

    list_menus.append(raw)

menus_raw = pd.concat(list_menus)
menus_raw.head()

,meal_id,meal_name,meal_type,is_kela,is_gluten_free,is_new,is_vegan
0,7609,"Lohta pesto & mustajuurta (L, G)",today's special,False,True,False,False
1,7607,"Broileria pekonikastikkeessa (L, G, KELA)",today's special,True,True,False,False
2,9039,BBQ-savutofuburger & raikasta nektariiniketsup...,today's special,False,False,True,False
3,6121,"Karamellisoitua possua (M,G,KELA)",today's special,True,True,False,False
4,6818,Filippiiniläiset kanavartaat & Hedelmäsalsaa (...,today's special,True,True,False,False


In [10]:
menus = menus_raw.copy()

# Decompose meal name and tag
pat1 = r"(\s?\(\s?[A-Z](\w|\,|\s)*\))"
pat2 = r"(\(|\)|\s)"

def _f_extract_tag(s: str):
    meal_name, tag = "", ""

    s = s.strip()
    out = re.findall(pat1, s)
    if len(out) < 1:
        meal_name = s
    else:
        meal_name = re.sub(pat1, "", s).strip()
        tag = out[0][0]
        tag = re.sub(pat2, '', tag)

    return pd.Series({
        'meal': meal_name,
        'tag': tag
    })
    
menus['meal_name'] = menus['meal_name'].apply(_f_extract_tag)['meal']

# Process meal_id
pat_meal_code = r"(\d*)\s*\/\s*(\d*)"

def _f_extract_meal_id(s):
    match s:
        case int():
            meal_id = s
        case str():
            meal_id = int(s.split('/')[0])

    return meal_id

menus['meal_id'] = menus['meal_id'].apply(_f_extract_meal_id)

# Remove duplicate meals
menus = menus.groupby('meal_id').first().reset_index()

# Row-wise processing 
meal_type_che_exac = ['fish', 'meat', 'today’s special', 'vegan-kpl', 'vegan-miscellaneous']
meal_type_phy = ['salad', 'baguettes']

def _f_process_row(row) -> pd.Series:
    out = {
        'meal_type': None,
        'schoolyear': None,
        'restaurant': None,
        'attributes': [],
    }

    # schoolyear
    out['schoolyear'] = '24-25'

    # meal_type
    meal_type = row.meal_type.strip()
    if meal_type in ['meat', 'fish']:
        out['meal_type'] = meal_type
    else:
        out['attributes'].append(meal_type)
    if row.is_vegan:
        out['meal_type'] = "vegan"

    # restaurant
    if meal_type in meal_type_che_exac:
        out['restaurant'] = 'che-exa-vik'
    elif meal_type in meal_type_phy:
        out['restaurant'] = 'phy'

    # other attributes
    if row.is_kela:
        out['attributes'].append("kela")
    if row.is_gluten_free:
        out['attributes'].append("gluten_free")

    return pd.Series(out)
processed = menus.apply(_f_process_row, axis=1)
menus = pd.concat(
    [
        menus[['meal_id', 'meal_name']],
        processed
    ],
    axis=1
)

menus.head()

,meal_id,meal_name,meal_type,schoolyear,restaurant,attributes
0,710,Broileri-Caesarsalaatti,None,24-25,phy,"[salad, kela]"
1,785,Meksikolainen uunimakkara,meat,24-25,che-exa-vik,"[kela, gluten_free]"
2,790,Uunimakkara ja sinappikastike,meat,24-25,che-exa-vik,"[kela, gluten_free]"
3,791,Carbonara-kastike & pastaa,meat,24-25,che-exa-vik,[gluten_free]
4,839,Chili-katkarapusalaatti,None,24-25,phy,"[salad, kela, gluten_free]"


### Compose list of baguettes

In [11]:
baguettes_raw = pd.read_excel("data/processed/phase_4/menus.xlsx", sheet_name="baguettes")

baguettes_raw.head()

,meal_id,meal,meal_type,co2,is_kela
0,1445,"Lounaspatonki, juusto",kasvis,0.79,False
1,1446,"Lounaspatonki, kalkkunatahna",kana,0.53,False
2,1447,"Lounaspatonki, kasvis",vegaani,0.35,True
3,1448,"Lounaspatonki, katkarapu",kala,1.04,True
4,1449,"Lounaspatonki, kinkku",liha,0.46,False


In [12]:
baguettes = baguettes_raw.copy()


# Remove unnecessary columns
baguettes.drop(columns='co2', inplace=True)

# Rename meal_type
baguettes['meal_type'] = baguettes['meal_type'].map({
    'kasvis': 'vegetarian',
    'kana': 'chicken',
    'vegaani': 'vegan',
    'kala': 'fish',
    'liha': 'meat'
})
baguettes['meal_name'] = baguettes['meal'].str.strip()


# Add other columns
baguettes['restaurant'] = 'phy'

baguettes['schoolyear'] = '24-25'

baguettes['attributes'] = baguettes['is_kela'].apply(lambda x: ['baguettes', 'kela'] if x else ['baguettes'])

# Remove redundant columns
baguettes = baguettes.drop(columns=['is_kela', 'meal'])

baguettes.head()

,meal_id,meal_type,meal_name,restaurant,schoolyear,attributes
0,1445,vegetarian,"Lounaspatonki, juusto",phy,24-25,[baguettes]
1,1446,chicken,"Lounaspatonki, kalkkunatahna",phy,24-25,[baguettes]
2,1447,vegan,"Lounaspatonki, kasvis",phy,24-25,"[baguettes, kela]"
3,1448,fish,"Lounaspatonki, katkarapu",phy,24-25,"[baguettes, kela]"
4,1449,meat,"Lounaspatonki, kinkku",phy,24-25,[baguettes]


In [13]:
menus = pd.concat([menus, baguettes], ignore_index=True)

menus.head()

,meal_id,meal_name,meal_type,schoolyear,restaurant,attributes
0,710,Broileri-Caesarsalaatti,None,24-25,phy,"[salad, kela]"
1,785,Meksikolainen uunimakkara,meat,24-25,che-exa-vik,"[kela, gluten_free]"
2,790,Uunimakkara ja sinappikastike,meat,24-25,che-exa-vik,"[kela, gluten_free]"
3,791,Carbonara-kastike & pastaa,meat,24-25,che-exa-vik,[gluten_free]
4,839,Chili-katkarapusalaatti,None,24-25,phy,"[salad, kela, gluten_free]"


## Merge `meals` with 2024-2025 menu

In [14]:
def coalesce(*args):
    for x in args:
        if x:
            return x
    return None

for meal in menus.itertuples():
    if meal.meal_id in meals:       # Look up with `meal_id` and found
        meal_info = meals[meal.meal_id]

        # # Handle corner case: 2 meals have same meal_id but different meal_
        map_mealname2id[meal.meal_name] = meal.meal_id
        
        meal_info['meal_type'] = coalesce(meal.meal_type, meal_info['meal_type'])
        meal_info['schoolyear'] = coalesce(meal.schoolyear, meal_info['schoolyear'])
        meal_info['restaurant'] = meal.restaurant
        meal_info['attributes'] = meal.attributes
    else:                           # Not found -> Look up with `meal_name` using Hamming distance
        # Closest meal in terms of Hamming distance is one having smallest distance and that distance is smaller than threshold
        best_meal = None
        best_dist = 1e10

        for name, meal_id in map_mealname2id.items():
            dist = hamming_distance(meal.meal_name, name)
            
            if dist <= THRES and dist < best_dist:
                best_dist = dist
                best_meal = name

        if best_meal:
            # Found meal having similar meal name -> Update meal info
            logger.debug(f"{meal.meal_name} ({meal.meal_id}) - {best_meal}")

            meal_info = meals[map_mealname2id[best_meal]]

            meal_info['meal_type'] = coalesce(meal.meal_type, meal_info['meal_type'])
            meal_info['schoolyear'] = coalesce(meal.schoolyear, meal_info['schoolyear'])
            meal_info['restaurant'] = meal.restaurant
            meal_info['attributes'] = meal.attributes

            map_mealname2id[meal.meal_name] = map_mealname2id[best_meal]
        else:
            # No found similar meal -> Add new meal
            logger.debug(f"New: {meal.meal_name}")

            map_mealname2id[meal.meal_name] = meal.meal_id
            meals[meal.meal_id] = {
                'meal_type': meal.meal_type,
                'schoolyear': meal.schoolyear,
                'restaurant': meal.restaurant,
                'attributes': meal.attributes,
            }

2025-01-30 12:40:20.341 | DEBUG    | __main__:<module>:32 - Mustajuurikeitto (1125) - Mustajuurikeitto
2025-01-30 12:40:20.344 | DEBUG    | __main__:<module>:32 - Kasviskorma (7202) - Kasviskorma
2025-01-30 12:40:20.357 | DEBUG    | __main__:<module>:32 - TexMex-siemenpyöryköitä ja Louisana-kastiketta (8983) - TexMex-siemenpyöryköitä ja Louisana-kastikett
2025-01-30 12:40:20.371 | DEBUG    | __main__:<module>:32 - Marokkolaiset kasvispihvit & Chermoula kastiketta (8985) - Marokkolaiset kasvispihvit & Chermoula kastik
2025-01-30 12:40:20.377 | DEBUG    | __main__:<module>:32 - Peas of Heaven chorizokeitto (8986) - Peas of Heaven chorizokeitto
2025-01-30 12:40:20.381 | DEBUG    | __main__:<module>:32 - Täysjyväkalafileetä & lime-korianteriremouladea (8988) - Täysjyväkalafileetä & lime-korianteriremoulad
2025-01-30 12:40:20.383 | DEBUG    | __main__:<module>:32 - Rapeaa ruiskalaa & piparjuurikastiketta (8989) - Rapeaa ruiskalaa & piparjuurikastiketta
2025-01-30 12:40:20.385 | DEBUG    | _

# Merge with meals from POS

## Process POS files

In [15]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = np.arange(df.shape[1])
    raw.append(df)


pos_raw = pd.concat(raw, ignore_index=True)
pos_raw.head()

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_45957/1467303410.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,0,1,2,3,4,5,6
0,2.1.2023,10:31,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
1,2.1.2023,10:32,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
2,2.1.2023,10:32,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
3,2.1.2023,10:35,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
4,2.1.2023,10:36,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",2,"1,8"


In [16]:
pos = pos_raw.copy()


# Rename columns
pos.columns = np.arange(pos.shape[1])
pos.rename(
    columns={
        0: 'date',
        1: 'time',
        2: 'restaurant',
        3: 'meal_type',
        4: 'meal',
        5: 'pcs',
        6: 'co2',
    },
    inplace=True
)


# Process date
pos['date'] = pd.to_datetime(pos['date'], format="%d.%m.%Y")



# Process meal_type
pos['meal_type'] = pos['meal_type'].map({
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken'
})


# Process meal
pos['meal'] = pos['meal'].str.strip()


# Map restaurant name
pos['restaurant'] = pos['restaurant'].map({
    '600 Chemicum': 'che-exa-vik', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'che-exa-vik', #'exactum'
    '570 Viikuna': 'che-exa-vik',
})


# Get pair of meal and meal_type
meals_pos = pos.groupby(['meal', 'meal_type']).last().reset_index()


# Extract schoolyear
meals_pos['schoolyear'] = meals_pos['date'].apply(lambda x: '24-25' if x >= pd.to_datetime("2024-09-01") else '23-24')


# Keep necessary columns
meals_pos = meals_pos[['meal', 'meal_type', 'restaurant', 'schoolyear']]


# Remove take away
meals_pos = meals_pos[~meals_pos['meal'].str.lower().str.contains('take away')]
meals_pos.rename(columns={'meal': 'meal_name'}, inplace=True)



meals_pos.head()

,meal_name,meal_type,restaurant,schoolyear
0,"""Butter"" luomukikhernekastiketta",vegan,che-exa-vik,24-25
1,Aurajuusto-pinaattilasagnettea,vegetarian,che-exa-vik,24-25
2,BBQ-Broilerikastiketta,chicken,che-exa-vik,24-25
3,Bangladeshilainen linssipata,vegan,che-exa-vik,23-24
4,Bar Myöhä Bbq-seitanbowl,vegan,che-exa-vik,24-25


In [17]:
meals_pos[meals_pos['meal_name'] == 'Panini, hot chili kana']

,meal_name,meal_type,restaurant,schoolyear
260,"Panini, hot chili kana",chicken,phy,24-25


## Merge meals from POS data with `meals`

In [18]:
THRES = 0.1

for meal_other in meals_pos.itertuples():
    # Find exact match via name
    if meal_other.meal_name in map_mealname2id:
        meal = meals[map_mealname2id[meal_other.meal_name]]

        meal['meal_type'] = coalesce(meal_other.meal_type, meal['meal_type'])
        meal['schoolyear'] = max(meal['schoolyear'], meal_other.schoolyear)
        meal['restaurant'] = coalesce(meal['restaurant'], meal_other.restaurant)

        continue
    

    # Find approximate match
    best_meal = None
    best_dist = 1e10
    
    for name, meal_id in map_mealname2id.items():
        dist = hamming_distance(meal_other.meal_name, name)
        
        if dist <= THRES and dist < best_dist:
            best_dist = dist
            best_meal = name

    # Handle 2 cases: found approximate match and not found
    if best_meal is None:
        # Not found similar meal -> Add new meal
        logger.debug(f"New: {meal_other.meal_name}")

        meal_id = max(max(meals.keys()), 90_000_000) + 1
        map_mealname2id[meal_other.meal_name] = meal_id
        meals[meal_id] = {
            'meal_type': meal_other.meal_type,
            'schoolyear': meal_other.schoolyear,
            'restaurant': meal_other.restaurant,
            'attributes': []
        }
    else:
        # Found meal having similar meal name -> Update meal info
        logger.debug(f"Approximate: {meal_other.meal_name} - {best_meal}")

        meal_info = meals[map_mealname2id[best_meal]]

        meal_info['meal_type'] = coalesce(meal_other.meal_type, meal_info['meal_type'])
        meal_info['schoolyear'] = coalesce(meal_other.schoolyear, meal_info['schoolyear'])
        meal_info['restaurant'] = meal_other.restaurant


        map_mealname2id[meal_other.meal_name] = map_mealname2id[best_meal]

2025-01-30 12:40:20.945 | DEBUG    | __main__:<module>:41 - Approximate: Aurajuusto-pinaattilasagnettea - Aurajuusto-pinaattilasagnette
2025-01-30 12:40:20.948 | DEBUG    | __main__:<module>:41 - Approximate: BBQ-Broilerikastiketta - BBQ-broilerikastiketta
2025-01-30 12:40:20.950 | DEBUG    | __main__:<module>:29 - New: Bangladeshilainen linssipata
2025-01-30 12:40:20.952 | DEBUG    | __main__:<module>:29 - New: Bataattipihvit BBQ-tomaattik.
2025-01-30 12:40:20.954 | DEBUG    | __main__:<module>:29 - New: Bataattipihvit,curry-minttukas
2025-01-30 12:40:20.956 | DEBUG    | __main__:<module>:41 - Approximate: Bataattipihvit,curry-minttusoi - Bataattipihvit,curry-minttukas
2025-01-30 12:40:20.958 | DEBUG    | __main__:<module>:29 - New: BeanItStroganoff
2025-01-30 12:40:20.961 | DEBUG    | __main__:<module>:29 - New: Broileri-nacho-salaattia
2025-01-30 12:40:20.963 | DEBUG    | __main__:<module>:29 - New: Broilerinfileetä vuohenjuustok
2025-01-30 12:40:20.965 | DEBUG    | __main__:<module

# Finalize `meals`

## Post-process `meals`

In [19]:
# Add `aliases`
for name, meal_id in map_mealname2id.items():
    if 'aliases' not in meals[meal_id]:
        meals[meal_id]['aliases'] = [name]
    else:
        meals[meal_id]['aliases'].append(name)



# Add `panini` to `attributes`
for name, meal_id in map_mealname2id.items():
    if 'panini' in name.lower():
        meals[meal_id]['attributes'].append('panini')

        logger.debug(f"panini: {meal_id}")


# Replace `restaurant`
for meal in meals.values():
    match (meal['restaurant']):
        case 'che-exa-vik':
            meal['restaurant'] = ['che', 'exa', 'vik']
        case 'phy':
            meal['restaurant'] = ['phy']


# Manually modify cases
del meals[6867]
del meals[6554]


df_meals = pd.DataFrame.from_dict(meals, orient='index').reset_index().rename(columns={'index': 'meal_id'})
df_meals.head()

2025-01-30 12:40:21.418 | DEBUG    | __main__:<module>:15 - panini: 6638
2025-01-30 12:40:21.418 | DEBUG    | __main__:<module>:15 - panini: 3524
2025-01-30 12:40:21.419 | DEBUG    | __main__:<module>:15 - panini: 3525
2025-01-30 12:40:21.419 | DEBUG    | __main__:<module>:15 - panini: 1513
2025-01-30 12:40:21.419 | DEBUG    | __main__:<module>:15 - panini: 1509
2025-01-30 12:40:21.419 | DEBUG    | __main__:<module>:15 - panini: 3211
2025-01-30 12:40:21.420 | DEBUG    | __main__:<module>:15 - panini: 4343
2025-01-30 12:40:21.420 | DEBUG    | __main__:<module>:15 - panini: 2906
2025-01-30 12:40:21.420 | DEBUG    | __main__:<module>:15 - panini: 1510
2025-01-30 12:40:21.420 | DEBUG    | __main__:<module>:15 - panini: 1512
2025-01-30 12:40:21.421 | DEBUG    | __main__:<module>:15 - panini: 6636
2025-01-30 12:40:21.421 | DEBUG    | __main__:<module>:15 - panini: 6096
2025-01-30 12:40:21.422 | DEBUG    | __main__:<module>:15 - panini: 1515
2025-01-30 12:40:21.422 | DEBUG    | __main__:<modu

,meal_id,meal_type,schoolyear,restaurant,attributes,aliases
0,9017,vegan,24-25,"[che, exa, vik]","[vegan-miscellaneous, kela]","[""Butter"" härkäpapua & pähkinää]"
1,7201,vegan,23-24,None,[],[2023 Härkäpu-sienilasagnette]
2,9032,vegan,23-24,None,[],[Appelisiini-luomukikhernecurrya]
3,9102,vegan,23-24,None,[],[Artisokkavugetteja & tuoretomaattisalsaa]
4,7010,vegetarian,24-25,"[che, exa, vik]",[],"[Aurajuusto-pinaattilasagnette, Aurajuusto-pin..."


## Unit tests

### Assert `map_mealname2id` contains all names

In [20]:
aliases = []
for alias in df_meals['aliases']:
    aliases.extend(alias)
aliases = set(aliases)

assert len(set(meals_pos['meal_name']).difference(aliases)) == 0
assert len(set(meal_list['meal']).difference(aliases)) == 0
assert len(set(menus['meal_name']).difference(aliases)) == 0

### Check `meal_type` in specific values

In [21]:
assert set(df_meals['meal_type'].unique()) == {'buffet', 'chicken', 'fish', 'meat', 'vegan', 'vegetarian'}

# Save dim

In [22]:
df_meals.to_parquet("data/processed/phase_4/dim_meals.parquet")